# **Project 2 - LLMs and Generative AI**

Below is a ready-to-use notebook outline for your assignment. Students will only need to change the prompts in designated cells. The notebook uses Hugging Face Transformers for open-source models, namely MedGemma.

## Install and Import Packages

In [1]:
# Package requirements for the notebook to run successfully
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 69.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [2]:
# Import necessary libraries for the project
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    pipeline
)
import torch

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## Load Generation Model and Tokenizer


**MedGemma Description:**

MedGemma is a collection of Gemma 3 variants that are trained for performance on medical text and image comprehension.

You can find more information here:
https://huggingface.co/unsloth/medgemma-4b-it-bnb-4bit.

In [3]:
# Define the name of the model to be loaded from Hugging Face
model_name = "unsloth/medgemma-4b-it-bnb-4bit"

# Print status message to let the user know which model is being loaded
print(f"🔄 Loading generation model: {model_name}")

# Load the language model (LLM) for text generation
generation_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32, # define the type of data to use if gpu is available
    device_map="auto"
)

# Load the tokenizer for the model
# The tokenizer is responsible for converting text to tokens the model can understand (and back to text)
generation_tokenizer = AutoTokenizer.from_pretrained(model_name)
generation_tokenizer.pad_token = generation_tokenizer.pad_token or generation_tokenizer.eos_token

🔄 Loading generation model: unsloth/medgemma-4b-it-bnb-4bit


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.23G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

Function to generate the output using the LLM defined above

In [4]:
def generate(messages):

    # Convert a list of chat messages into a single string using the model's chat template.
    # This prepares the conversation in the format the model expects (e.g., <user>, <assistant> tags).
    text = generation_tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

    # Tokenize the input text and prepare it for the model.
    inputs = generation_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=1000).to(generation_model.device)

    # Generate output using the model
    with torch.inference_mode():
        generation = generation_model.generate(
            **inputs, max_new_tokens=300, do_sample=False,
            pad_token_id=generation_tokenizer.eos_token_id
        )

    # Decode the model's output, transforming the token IDs back into a human-readable string.
    output = generation_tokenizer.decode(generation[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    return output

## Load Notes

**Note 1:** English clinical note

In [5]:
note = """The patient is a 72-year-old female with a past medical history of atrial fibrillation, chronic kidney disease stage 3, hypertension, and osteoarthritis.
She presented to the ED with complaints of generalized weakness, lightheadedness, and two episodes of syncope over the past 24 hours.

On exam, she was hypotensive (BP 88/56) and bradycardic with an irregularly irregular pulse. Labs showed elevated BUN and creatinine, consistent with worsening CKD, and mild hyperkalemia.
ECG revealed atrial fibrillation with slow ventricular response. Chest X-ray showed no acute pulmonary findings.
The patient was admitted to telemetry for further monitoring. Her beta-blocker (metoprolol) was held, and nephrology was consulted for worsening renal function. C
ardiology was consulted for bradyarrhythmia management. A temporary transvenous pacemaker was placed.

She was monitored closely, and after stabilization, underwent permanent pacemaker implantation on hospital day 3. Post-op course was uncomplicated.
Renal function improved with IV fluids and adjustment of antihypertensive medications.

Discharge medications included: apixaban, lisinopril (dose reduced), and acetaminophen for pain. She was advised to follow up with cardiology and nephrology within one week.
Discharged home with stable vital signs and improved energy levels."""

**Note 2:** Portuguese clinical note

In [6]:
note_pt = """1.a consulta
NHC 118441566
H, 67 anos, de Sesimbra, mestre pesca reformado (desde 58 anos)
Sempre trabalhou de noite e dormiu de dia. Dormia cerca de 3 h.
Desde que se reformou tem dificuldade em dormir. Dorme 3-4 h
Tem uma empresa. Por vezes ansioso
Faz alprazolan 0,25 mg ao jantar. Deita-se 21h-22h. Leva cerca de 2 horas para adormecer. Dorme 3-4 h. ACorda e não sabe porquê. Fica na cama e depois levanta-se.
Antes
Já fez mirtazapina 15 mg, durante 1 Mês e acha que não teve efeito terapeutico
DRC por provável pielonefrite crónica litiásica (hiperuricémia?), inicio de hemodiálise em julho 2002, Transplante renal de dador vivo a 17/09/2002. Reinicio de hemodiálise por disfunção crónica do enxerto a 17/08/2016
HTA secundária
Hábitos tabágicos suspensos 2002

Adenocrcinoma da prostata, submetido a prostatectomia total em janeiro de 2018

Tumor Septissémia por bactéria provocada por um peixe... em set 2018. Ferida no MS dto...

Medicação atual: amlodipina e concor . Alprazolam

Refere sensação de calor nos membros inferiores que melhora com movimento, piora no final do dia, por vezes dificultam o adormecer

Roncopatia. Sem apneias identificadas.

Xavier Mourato OM 52847"""

## **Exercise 1: Summarization**

<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h4>📝 Exercise 1.1 - Interpretation of the Clinical Note Summary
  <p>Consider the following example for summarize the clinical note.
  
  What can you say about the summary generated? Is the model hallucinating?
  </p>
</div>


In [7]:
# This variable will be our prompt for summarization.
messages = [
    {
        "role": "system",
        "content": "You are a helpful medical assistant."
    },
    {
        "role": "user",
        "content": f"""Summarize the following clinical note:\n\nCLINICAL NOTE:\n{note}"""
    }
]

# Here we will generate an output for the prompt using the generator function
output = generate(messages)
print(output)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


A 72-year-old female with a history of atrial fibrillation, chronic kidney disease (CKD), hypertension, and osteoarthritis presented to the ED with weakness, lightheadedness, and syncope. She was found to be hypotensive and bradycardic with atrial fibrillation. Labs showed elevated BUN and creatinine, and mild hyperkalemia. An ECG showed atrial fibrillation with slow ventricular response. A temporary pacemaker was placed, and a permanent pacemaker was implanted on hospital day 3. Her renal function improved with IV fluids and medication adjustments. She was discharged home with stable vital signs, improved energy levels, and follow-up appointments with cardiology and nephrology.



In [8]:
note

'The patient is a 72-year-old female with a past medical history of atrial fibrillation, chronic kidney disease stage 3, hypertension, and osteoarthritis.\nShe presented to the ED with complaints of generalized weakness, lightheadedness, and two episodes of syncope over the past 24 hours.\n\nOn exam, she was hypotensive (BP 88/56) and bradycardic with an irregularly irregular pulse. Labs showed elevated BUN and creatinine, consistent with worsening CKD, and mild hyperkalemia.\nECG revealed atrial fibrillation with slow ventricular response. Chest X-ray showed no acute pulmonary findings.\nThe patient was admitted to telemetry for further monitoring. Her beta-blocker (metoprolol) was held, and nephrology was consulted for worsening renal function. C\nardiology was consulted for bradyarrhythmia management. A temporary transvenous pacemaker was placed.\n\nShe was monitored closely, and after stabilization, underwent permanent pacemaker implantation on hospital day 3. Post-op course was un

In [9]:
# This variable will be our prompt for summarization.
messages = [
    {
        "role": "system",
        "content": "You are a helpful medical assistant that understands portuguese."
    },
    {
        "role": "user",
        "content": f"""Summarize the following clinical note in portuguese:\n\nCLINICAL NOTE:\n{note_pt}"""
    }
]

# Here we will generate an output for the prompt using the generator function
output = generate(messages)
print(output)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Aqui está um resumo da nota clínica em português:

**Resumo:**

Paciente masculino, 67 anos, de Sesimbra, com histórico de pesca e trabalho noturno. Apresenta insônia há algum tempo, com dificuldade para adormecer e acordar durante a noite. Utiliza alprazolam (0,25 mg) à noite, mas o sono é interrompido. Anteriormente, tentou mirtazapina sem sucesso. Possui histórico de pielonefrite crônica, transplante renal em 2002, reinício de hemodiálise em 2016, hipertensão arterial secundária, tabagismo interrompido em 2002, adenocarcinoma da próstata tratado por prostatectomia em 2018 e tumor septissêmica por bactéria de peixe em 2018.  Atualmente, toma amlodipina e concor, além de alprazolam. Relata sensação de calor nos membros inferiores que melhora com movimento e piora no final do dia, dificultando o sono. Apresenta roncopatia sem apneias identificadas.



In [10]:
# This variable will be our prompt for summarization.
messages = [
    {
        "role": "system",
        "content": "You are a helpful medical assistant, that is proeficient in portuguese."
    },
    {
        "role": "user",
        "content": f"""Summarize the following clinical note:\n\nCLINICAL NOTE:\n{note_pt}"""
    }
]

# Here we will generate an output for the prompt using the generator function
output = generate(messages)
print(output)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Aqui está um resumo da nota clínica:

**Paciente:** Homem de 67 anos, de Sesimbra, ex-mestre pescador, reformado há 5 anos.

**Problema Principal:** Insónia (dificuldade em dormir) desde a reforma, com duração de 3-4 horas por noite.

**Histórico:**

*   Trabalhou à noite e dormiu durante o dia por muitos anos.
*   Ansiedade.
*   Tomava alprazolam (0,25 mg) ao jantar, mas leva cerca de 2 horas para adormecer.
*   Antes tomou mirtazapina (15 mg) por um mês, sem melhora.
*   Histórico de pielonefrite crónica litiásica (hiperuricémia?), com início de hemodiálise em 2002, seguido de transplante renal em 2002 e reinício da hemodiálise em 2016 devido a disfunção do enxerto.
*   HTA secundária.
*   Tabagista (parou em 2002).
*   Adenocarcinoma da próstata, submetido a prostatectomia total em 2018.
*   Tumor septissémico por bactéria de peixe em 2018.
*   Ferida no MS (m


In [11]:
# This variable will be our prompt for summarization.
messages = [
    {
        "role": "system",
        "content": "You are a helpful medical assistant, that is proeficient in portuguese."
    },
    {
        "role": "user",
        "content": f"""Summarize the following clinical note, in a paragraph:\n\nCLINICAL NOTE:\n{note_pt}"""
    }
]

# Here we will generate an output for the prompt using the generator function
output = generate(messages)
print(output)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


O paciente, um homem de 67 anos, mestre pescador reformado, apresenta insônia crônica desde a aposentadoria, com dificuldades para adormecer e acordar durante a noite. Ele toma alprazolam à noite e relata sensações de calor nos membros inferiores que melhoram com movimento e pioram no final do dia, dificultando o sono. O paciente tem histórico de hipertensão secundária, pielonefrite crônica, transplante renal, adenocarcinoma da próstata tratado por prostatectomia, e um tumor septissêmico por bactéria de peixe. Ele também relata roncopatia e usa amlodipina e concor.



In [12]:
note_pt

'1.a consulta\nNHC 118441566\nH, 67 anos, de Sesimbra, mestre pesca reformado (desde 58 anos)\nSempre trabalhou de noite e dormiu de dia. Dormia cerca de 3 h.\nDesde que se reformou tem dificuldade em dormir. Dorme 3-4 h\nTem uma empresa. Por vezes ansioso\nFaz alprazolan 0,25 mg ao jantar. Deita-se 21h-22h. Leva cerca de 2 horas para adormecer. Dorme 3-4 h. ACorda e não sabe porquê. Fica na cama e depois levanta-se.\nAntes\nJá fez mirtazapina 15 mg, durante 1 Mês e acha que não teve efeito terapeutico\nDRC por provável pielonefrite crónica litiásica (hiperuricémia?), inicio de hemodiálise em julho 2002, Transplante renal de dador vivo a 17/09/2002. Reinicio de hemodiálise por disfunção crónica do enxerto a 17/08/2016\nHTA secundária\nHábitos tabágicos suspensos 2002\n\nAdenocrcinoma da prostata, submetido a prostatectomia total em janeiro de 2018\n\nTumor Septissémia por bactéria provocada por um peixe... em set 2018. Ferida no MS dto...\n\nMedicação atual: amlodipina e concor . Alpra

<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h4>📝 Exercise 1.2 - Structured Clinical Notes
  <p>What if we want to structure the clinical Note?
  
  Change the prompt presented in the example before to have the summary of the clinical note in a structured format.
  
  Example of a structured clinical note:
  * Demographics:
  * History:
  * Diagnosis:
  * ...
  </p>
</div>

In [13]:
# This variable will be our prompt for summarization.
messages = [
    {
        "role": "system",
        "content": "You are a helpful medical assistant."
    },
    {
        "role": "user",
        "content": f"""Summarize the following clinical note, in key conpects:\n\nCLINICAL NOTE:\n{note}"""
    }
]

# Here we will generate an output for the prompt using the generator function
output = generate(messages)
print(output)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Here's a summary of the clinical note, focusing on key aspects:

*   **Patient:** 72-year-old female with significant comorbidities: atrial fibrillation, chronic kidney disease (CKD) stage 3, hypertension, and osteoarthritis.
*   **Presenting Symptoms:** Generalized weakness, lightheadedness, and syncope (2 episodes in 24 hours).
*   **Exam Findings:** Hypotension (BP 88/56), bradycardia with irregularly irregular pulse, elevated BUN and creatinine (worsening CKD), mild hyperkalemia.
*   **ECG:** Atrial fibrillation with slow ventricular response.
*   **Interventions:**
    *   Admitted to telemetry for monitoring.
    *   Beta-blocker (metoprolol) held.
    *   Nephrology consulted for worsening renal function.
    *   Cardiology consulted for bradyarrhythmia management.
    *   Temporary transvenous pacemaker placed.
    *   Permanent pacemaker implanted on hospital day 3.
*   **Management:**
    *   IV fluids and antihypertensive medication adjustments improved renal function.
*   *

## **Exercise 2: Question-Answering**
We can use the LLM to answer specific questions regarding the clinical note.

<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h4>📝 Exercise 2.1 - Patient status on Exam
  <p>How was the patient feeling in the moment of the exame?

  Change the prompt and interpret the output.
  </p>
</div>

In [14]:
# This variable will be our prompt for summarization.
messages = [
    {
        "role": "system",
        "content": "You are a helpful medical assistant."
    },
    {
        "role": "user",
        "content": f"""How was the patient feeling in the moment of the exame in the following clinical note:\n\nCLINICAL NOTE:\n{note}"""
    }
]

# Here we will generate an output for the prompt using the generator function
output = generate(messages)
print(output)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In the moment of the exam, the patient was **hypotensive (BP 88/56) and bradycardic with an irregularly irregular pulse.**



In [15]:
# This variable will be our prompt for summarization.
messages = [
    {
        "role": "system",
        "content": "You are a helpful medical assistant that understands and answers in portuguese."
    },
    {
        "role": "user",
        "content": f"""How was the patient feeling in the moment of the exame in the following clinical note:\n\nCLINICAL NOTE:\n{note_pt}"""
    }
]

# Here we will generate an output for the prompt using the generator function
output = generate(messages)
print(output)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


No momento do exame, o paciente relatou sentir **ansiedade** e **sensação de calor nos membros inferiores**. A sensação de calor melhorava com movimento e piorava no final do dia, dificultando o adormecer.



<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h4>📝 Exercise 2.2 - Discharge Medication
  <p>Which drugs are included in the discharge medications?

  Change the prompt and interpret the output. Try include the doses if they are available.
  </p>
</div>

In [16]:
#This variable will be our prompt for summarization.
messages = [
    {
        "role": "system",
        "content": "You are a helpful medical assistant that understands portuguese."
    },
    {
        "role": "user",
        "content": f"""Summarize the following clinical,Which drugs are included in the discharge medications :\n\nCLINICAL NOTE:\n{note_pt}"""
    }
]

# Here we will generate an output for the prompt using the generator function
output = generate(messages)
print(output)


The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Based on the clinical note, the following drugs are included in the discharge medications:

*   **Amlodipina**
*   **Concor**
*   **Alprazolam**


In [17]:
#This variable will be our prompt for summarization.
messages = [
    {
        "role": "system",
        "content": "You are a helpful medical assistant that understands portuguese."
    },
    {
        "role": "user",
        "content": f"""Summarize the following clinical,Which drugs are included in the discharge medications :\n\nCLINICAL NOTE:\n{note}"""
    }
]

# Here we will generate an output for the prompt using the generator function
output = generate(messages)
print(output)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The discharge medications for this patient are:

*   **Apixaban:** An anticoagulant for stroke prevention in atrial fibrillation.
*   **Lisinopril:** An ACE inhibitor for blood pressure control, with the dose reduced.
*   **Acetaminophen:** For pain relief.



## **Exercise 3 - Entity Extraction**

<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h4>📝 Exercise 3.1 - Procedures Extraction
  <p>Still using the same LLM, find the procedures mentioned in the clinical note?

  Change the prompt and interpret the output.
  </p>
</div>

In [18]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

note = """The patient is a 72-year-old female with a past medical history of atrial fibrillation, chronic kidney disease stage 3, hypertension, and osteoarthritis.
She presented to the ED with complaints of generalized weakness, lightheadedness, and two episodes of syncope over the past 24 hours.

On exam, she was hypotensive (BP 88/56) and bradycardic with an irregularly irregular pulse. Labs showed elevated BUN and creatinine, consistent with worsening CKD, and mild hyperkalemia.
ECG revealed atrial fibrillation with slow ventricular response. Chest X-ray showed no acute pulmonary findings.
The patient was admitted to telemetry for further monitoring. Her beta-blocker (metoprolol) was held, and nephrology was consulted for worsening renal function.
Cardiology was consulted for bradyarrhythmia management. A temporary transvenous pacemaker was placed.

She was monitored closely, and after stabilization, underwent permanent pacemaker implantation on hospital day 3. Post-op course was uncomplicated.
Renal function improved with IV fluids and adjustment of antihypertensive medications.

Discharge medications included: apixaban, lisinopril (dose reduced), and acetaminophen for pain. She was advised to follow up with cardiology and nephrology within one week.
Discharged home with stable vital signs and improved energy levels."""
# Define the name of the model to be loaded from Hugging Face
model_name = "unsloth/medgemma-4b-it-bnb-4bit"

# Print status message to let the user know which model is being loaded
print(f"🔄 Loading generation model: {model_name}")

# Load the language model (LLM) for text generation
generation_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32, # define the type of data to use if gpu is available
    device_map="auto"
)

# Load the tokenizer for the model
# The tokenizer is responsible for converting text to tokens the model can understand (and back to text)
generation_tokenizer = AutoTokenizer.from_pretrained(model_name)
generation_tokenizer.pad_token = generation_tokenizer.pad_token or generation_tokenizer.eos_token

def generate(messages):

    # Convert a list of chat messages into a single string using the model's chat template.
    # This prepares the conversation in the format the model expects (e.g., <user>, <assistant> tags).
    text = generation_tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

    # Tokenize the input text and prepare it for the model.
    inputs = generation_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=1000).to(generation_model.device)

    # Generate output using the model
    with torch.inference_mode():
        generation = generation_model.generate(
            **inputs, max_new_tokens=300, do_sample=False,
            pad_token_id=generation_tokenizer.eos_token_id
        )

    # Decode the model's output, transforming the token IDs back into a human-readable string.
    output = generation_tokenizer.decode(generation[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    return output
messages = [
    {
        "role": "system",
        "content": "You are a medical."
    },
    {
        "role": "user",
        "content": f"""Find the procedures mentioned in the clinical note:\n\nCLINICAL NOTE:\n{note}"""
    }
]

# Here we will generate an output for the prompt using the generator function
output = generate(messages)
print(output)

🔄 Loading generation model: unsloth/medgemma-4b-it-bnb-4bit


The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The following procedures were mentioned in the clinical note:

*   **Temporary transvenous pacemaker placement**
*   **Permanent pacemaker implantation**
*   **ECG**
*   **Chest X-ray**
*   **IV fluids administration**
*   **Adjustment of antihypertensive medications**
*   **Blood tests (BUN, creatinine, potassium)**
*   **Consultation with nephrology**
*   **Consultation with cardiology**
*   **Medication adjustments (holding metoprolol, reducing lisinopril)**
*   **Discharge home**


<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h4>📝 Exercise 3.2 - Conditions Extraction for clinical note in portuguese
  <p>Still using the same LLM, find the conditions mentioned in the clinical note written in portuguese?

  Change the prompt and interpret the output. The portuguese note is saved in the variable 'note_pt'.
  </p>
</div>

In [19]:
messages = [
    {
        "role": "system",
        "content": "Você é um assistente médico útil."
    },
    {
        "role": "user",
        "content": f"""Extrai as condições clínicas mencionadas na seguinte nota médica:\n\nNOTA CLÍNICA:\n{note_pt}"""
    }
]

output = generate(messages)
print(output)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


As condições clínicas mencionadas na nota médica são:

*   **Insónia:** Dificuldade em dormir, com duração de 3-4 horas.
*   **Ansiedade:** Sentimentos de ansiedade.
*   **Pielonefrite crónica litiásica (hiperuricémia?):** Possível infecção renal crônica com cálculos.
*   **Disfunção crónica do enxerto:** Problemas com o transplante renal.
*   **Hipertensão arterial (HTA) secundária:** HTA causada por outra condição médica.
*   **Adenocarcinoma da próstata:** Câncer de próstata, tratado por prostatectomia total.
*   **Septissémia por bactéria provocada por um peixe:** Infecção bacteriana no sangue, possivelmente relacionada a um peixe.
*   **Roncopatia:** Distúrbio respiratório durante o sono.
*   **Sensação de calor nos membros inferiores:** Sensação de calor nos membros inferiores.
*   **Dificuldade em adormecer:** Dificuldade em adormecer.
*   **Disfunção erétil:** Implícito pela dificuldade em adormecer.
*   **Hiperuricémia:** Aumento do ácido úrico no sangue.
*   **Disfunção do en

<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h4>📝 Exercise 3.3 - Named-Entity Recognition (NER) Model
  <p> These type of models are trained to identify entities in the clinical notes. Below we show how to use a NER model to extract entities, in this case conditions, from the clinical note.

  Considering the conditions identified, which are the main differences comparing to the output generated in the previous exercise?
  </p>
</div>

In [20]:
note_pt = """1.a consulta
NHC 118441566
H, 67 anos, de Sesimbra, mestre pesca reformado (desde 58 anos)
Sempre trabalhou de noite e dormiu de dia. Dormia cerca de 3 h.
Desde que se reformou tem dificuldade em dormir. Dorme 3-4 h
Tem uma empresa. Por vezes ansioso
Faz alprazolan 0,25 mg ao jantar. Deita-se 21h-22h. Leva cerca de 2 horas para adormecer. Dorme 3-4 h. ACorda e não sabe porquê. Fica na cama e depois levanta-se.
Antes
Já fez mirtazapina 15 mg, durante 1 Mês e acha que não teve efeito terapeutico
DRC por provável pielonefrite crónica litiásica (hiperuricémia?), inicio de hemodiálise em julho 2002, Transplante renal de dador vivo a 17/09/2002. Reinicio de hemodiálise por disfunção crónica do enxerto a 17/08/2016
HTA secundária
Hábitos tabágicos suspensos 2002

Adenocrcinoma da prostata, submetido a prostatectomia total em janeiro de 2018

Tumor Septissémia por bactéria provocada por um peixe... em set 2018. Ferida no MS dto...

Medicação atual: amlodipina e concor . Alprazolam

Refere sensação de calor nos membros inferiores que melhora com movimento, piora no final do dia, por vezes dificultam o adormecer

Roncopatia. Sem apneias identificadas.

Xavier Mourato OM 52847"""
entity_extractor = pipeline('ner', model='portugueseNLP/medialbertina_pt-pt_900m_NER', aggregation_strategy="simple")
entities = entity_extractor(note_pt)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.54G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.45M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/971 [00:00<?, ?B/s]

Device set to use cuda:0


In [21]:
diagnosis = [annotation['word'] for annotation in entities if annotation['entity_group'] == 'Diagnostico']
diagnosis

['DRC',
 'pielonefrite crónica litiásica (hiperuricémia?',
 'disfunção crónica do enxerto',
 'HTA secundária',
 'Adenocrcinoma da prostata',
 'Tumor Septissémia',
 'bactéria provocada por um peixe...',
 'Ferida no MS dto',
 'Roncopatia',
 'apneias']

**Model Description:**

MediAlbertina PT-PT 900M NER was created through fine-tuning of MediAlbertina PT-PT 900M on real European Portuguese EMRs that have been hand-annotated for the following entities:

- Diagnostico (D): All types of diseases and conditions following the ICD-10-CM guidelines.
- Sintoma (S): Any complaints or evidence from healthcare professionals indicating that a patient is experiencing a medical condition.
Medicamento (M): Something that is administrated to the patient (through any route), including drugs, specific food/drink, vitamins, or blood for transfusion.
- Dosagem (D): Dosage and frequency of medication administration.
ProcedimentoMedico (PM): Anything healthcare professionals do related to patients, including exams, moving patients, administering something, or even surgeries.
- SinalVital (SV): Quantifiable indicators in a patient that can be measured, always associated with a specific result. Examples include cholesterol levels, diuresis, weight, or glycaemia.
- Resultado (R): Results can be associated with Medical Procedures and Vital Signs. It can be a numerical value if something was measured (e.g., the value associated with blood pressure) or a descriptor to indicate the result (e.g., positive/negative, functional).
- Progresso (P): Describes the progress of patient’s condition. Typically, it includes verbs like improving, evolving, or regressing and mentions to patient’s stability.

https://huggingface.co/portugueseNLP/medialbertina_pt-pt_900m_NER

# Comparação:
`unsloth/medgemma-4b-it-bnb-4bit` vs Modelos de Named Entity Recognition (NER)

Para netender melhor as principais diferenças entre um **modelo de linguagem médica** (LLM) e um **modelo de reconhecimento de entidades nomeadas (NER)**, temos:

---

##  1. Propósito e Função

| Característica        | `unsloth/medgemma-4b-it`                               | Modelo NER                                  |
|-----------------------|--------------------------------------------------------|----------------------------------------------|
| **Tipo**              | Modelo de linguagem (LLM) médico                      | Classificador de tokens                      |
| **Objetivo principal**| Responder perguntas, gerar textos, sumarizar          | Identificar e rotular entidades no texto     |
| **Compreensão de contexto** |  Completa, com capacidade de inferência         |  Limitada ao nível de palavra/token         |

---

##  2. Formato de Input e Output

| Aspecto        | `medgemma-4b-it`                                             | Modelo NER                                  |
|----------------|--------------------------------------------------------------|----------------------------------------------|
| **Input**     | Prompt do tipo: `"Quais são as causas da hipertensão?"`    | Texto cru ou tokenizado                      |
| **Output**       | Texto livre: `"Hipertensão é causada por fatores como..."` | Estrutura com rótulos: `{"diabetes": "DOENÇA"}` |
| **Granularidade** | Nível de sentença ou parágrafo                          | Nível de palavra/token                       |

---

##  3. Quando usar

| Quando usar                     | `medgemma-4b-it` | Modelo NER |
|--------------------------------|------------------|------------|
| Sumarização clínica            | sim               | não         |
| Perguntas e respostas médicas  | sim               | não         |
| Extração de entidades (doenças, fármacos) |  Possível com prompt |  Otimizado |
| Integração com bases estruturadas (EHR/EMR) |  Limitado |  Simples e direto |

---

##  4. Diferenças Técnicas

| Recurso              | `medgemma-4b-it`                        | Modelo NER                            |
|----------------------|-----------------------------------------|----------------------------------------|
| **Tipo de modelo**   | LLM decoder com fine-tuning instrucional| Classificador token-level (BERT, etc.) |
| **Treinamento**      | Instruções médicas + textos clínicos     | Corpora anotados (ex: BC5CDR, i2b2)    |
| **Velocidade de inferência** | Mais lenta (especialmente em 4-bit) | Muito rápida                          |
| **Consistência na saída** | Variável (dependente do prompt)       | Alta (resultado por token)            |

---

##  De forma prática, podemos visualizar com um exemplo:

**Input:**  
`"O paciente foi diagnosticado com diabetes tipo 2 e prescrito metformina."`

| Modelo                 | Saída                                                                 |
|------------------------|----------------------------------------------------------------------|
| `unsloth/medgemma-4b-it` | `"Diabetes tipo 2 é um distúrbio metabólico crônico tratado com metformina."` |
| Modelo NER             | `{"diabetes tipo 2": "DOENÇA", "metformina": "FÁRMACO"}`             |

---

##  Portanto:

| Categoria                         | `medgemma-4b-it` | Modelo NER |
|----------------------------------|------------------|------------|
| Geração de texto natural         | sim               | não         |
| Extração estruturada de entidades|  Com limitações| Sem limitações         |
| Compreensão de linguagem médica  | sim               | Limitada |
| Anotação tokenizada              | sim               | não         |
| Ideal para pipelines clínicos    | Não               | Sim         |

---
Em resumo:
>  **Podemos usar NER para tarefas de extração estruturada**.  
>  **E `medgemma-4b-it` para raciocínio, respostas médicas e geração de texto clínico.**

---
